## 03. 차트 프로토타입

Plotly로 산점도(개별 거래)·선 그래프(월별 중위가)·막대(거래량)를 단계별로 그려봅니다.
마포래미안푸르지오 84㎡(30평대)를 예시 데이터로 사용합니다.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = str(Path.cwd().resolve().parents[2])
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from projects.part10_price_volume.dashboard.preprocessing import (
    get_individual_trades, get_monthly_summary
)
from projects.part10_price_volume.dashboard.charts import (
    create_scatter_chart, create_line_chart
)

# 공통 조회 조건
QUERY_PARAMS = dict(
    시도='서울특별시',
    시군구='마포구',
    단지=['마포래미안푸르지오'],
    평형대=['30평대'],
    시작년월=202000,
    종료년월=202509,
)

### 1. 데이터 조회

In [ ]:
# 개별 거래 데이터 (산점도용)
data = get_individual_trades(**QUERY_PARAMS)
print(f"매매 {len(data.get('매매', []))}건 / 전세 {len(data.get('전세', []))}건")

In [ ]:
# 월별 집계 데이터 (선 그래프용)
monthly = get_monthly_summary(**QUERY_PARAMS)
print(f"매매 {len(monthly.get('매매', []))}개월 / 전세 {len(monthly.get('전세', []))}개월")

### 2. 산점도 — 개별 거래가 (평형대별 색상, jitter 적용)

같은 월 거래가 겹치지 않도록 x축에 ±12일 jitter를 추가한다.
평형대별 색상으로 면적 구간을 구분한다.

In [ ]:
fig = create_scatter_chart(data)
fig.show()

### 3. 선 그래프 — 월별 중위가 추이

매매·전세 중위가를 선 그래프로 그린다. 하단에 거래량 막대를 배치한다.

In [ ]:
fig = create_line_chart(monthly)
fig.show()

### 4. 데이터로 읽는 가격 흐름

월별 중위가 데이터를 직접 조회하여 주요 전환점을 확인한다.

In [ ]:
import pandas as pd

df = monthly.get('매매', pd.DataFrame())
if not df.empty:
    # 연도별 중위가 요약
    df['연도'] = df['계약년월'] // 100
    yearly = df.groupby('연도')['중위가_억'].median().reset_index()
    yearly.columns = ['연도', '연간_중위가_억']
    display(yearly)

In [ ]:
if not df.empty:
    # 연도별 등락률 계산
    yearly['전년대비_변화율'] = yearly['연간_중위가_억'].pct_change() * 100
    print(yearly.to_string(index=False))

### 5. 여러 단지 비교 — 마포구 30평대

단지를 지정하지 않으면 시군구 전체 데이터를 조회한다.

In [ ]:
# 마포구 30평대 전체 월별 추이
monthly_mapo = get_monthly_summary(
    시도='서울특별시',
    시군구='마포구',
    평형대=['30평대'],
    시작년월=202000,
    종료년월=202509,
)

fig = create_line_chart(monthly_mapo)
fig.update_layout(title='마포구 30평대 — 매매·전세 중위가 추이')
fig.show()